# Regression Baselines

## Imports

In [9]:
import os
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt

import ipywidgets as widgets
from IPython.display import display

## Load Data

In [2]:
base_dir = os.path.join(os.getcwd(), '..')
processed_data_dir = os.path.join(base_dir, 'processed_data')

open_meteo_filepath = os.path.join(processed_data_dir, 'open_meteo_processed_data.csv')
open_meteo_df = pd.read_csv(open_meteo_filepath, parse_dates=['datetime'])

ausgrid_filepath = os.path.join(processed_data_dir, 'ausgrid_processed_data.csv')
ausgrid_df = pd.read_csv(ausgrid_filepath, parse_dates=['datetime'])

## Make Datasets

In [3]:
# Merge weather data onto each smart-meter reading, then split chronologically
data = ausgrid_df.merge(open_meteo_df, on="datetime", how="left").sort_values("datetime")

# Split into training and testing data
cutoff = data["datetime"].drop_duplicates().sort_values().iloc[
    int(0.8 * data["datetime"].nunique())]

train_df = data[data["datetime"] < cutoff]
test_df = data[data["datetime"] >= cutoff]

## Average Baseline

In [6]:
# Half-hour slot within the day (0–47)
train_month = train_df["datetime"].dt.month
train_slot = train_df["datetime"].dt.hour * 2 + train_df["datetime"].dt.minute // 30

test_month = test_df["datetime"].dt.month
test_slot = test_df["datetime"].dt.hour * 2 + test_df["datetime"].dt.minute // 30

# Average historical PV generation by customer, month, and half-hour slot
monthly_profile = (
    pd.DataFrame({
        "Customer": train_df["Customer"].to_numpy(),
        "month": train_month.to_numpy(),
        "slot": train_slot.to_numpy(),
        "Pgen": train_df["Pgen"].to_numpy(),
    })
    .groupby(["Customer", "month", "slot"])["Pgen"]
    .mean()
)

# Fallback profile for month/slot combinations not present in training data
slot_profile = (
    pd.DataFrame({
        "Customer": train_df["Customer"].to_numpy(),
        "slot": train_slot.to_numpy(),
        "Pgen": train_df["Pgen"].to_numpy(),
    })
    .groupby(["Customer", "slot"])["Pgen"]
    .mean()
)

test_keys = pd.MultiIndex.from_arrays(
    [test_df["Customer"].to_numpy(), test_month.to_numpy(), test_slot.to_numpy()]
)
fallback_keys = pd.MultiIndex.from_arrays(
    [test_df["Customer"].to_numpy(), test_slot.to_numpy()]
)

predictions = monthly_profile.reindex(test_keys)
fallback_predictions = slot_profile.reindex(fallback_keys).to_numpy()
predictions = predictions.to_numpy()
predictions = np.where(pd.isna(predictions), fallback_predictions, predictions)

test_df["Pgen_pred"] = predictions

### Plot mean predictor results

In [8]:
test_plot_df = test_df.set_index("datetime").sort_index()
days = sorted(test_df["datetime"].dt.date.unique())
customers = sorted(test_df["Customer"].unique())

day_dropdown = widgets.Dropdown(options=days, description="Day:")
customer_dropdown = widgets.Dropdown(options=customers, description="Customer:")

def plot_customer_day(day, customer):
    start = pd.Timestamp(day)
    end = start + pd.Timedelta(days=1)
    day_data = test_plot_df.loc[start:end]
    customer_data = day_data[day_data["Customer"] == customer]

    ax = customer_data[["Pgen", "Pgen_pred"]].plot(figsize=(14, 5), linewidth=1)
    ax.set(
        title=f"Actual vs. Predicted PV Generation — Customer {customer}, {day}",
        xlabel="Time",
        ylabel="PV generation",
    )
    ax.legend(["Actual", "Predicted"])
    plt.tight_layout()
    plt.show()

controls = widgets.HBox([day_dropdown, customer_dropdown])
output = widgets.interactive_output(
    plot_customer_day,
    {"day": day_dropdown, "customer": customer_dropdown},
)
display(controls, output)

Output()

In [ ]:
# Compare actual and predicted generation across the test set
scatter_df = test_df[["Pgen", "Pgen_pred"]].dropna()
x = scatter_df["Pgen_pred"].to_numpy()
y = scatter_df["Pgen"].to_numpy()

# Best-fit line and R², calculated using all valid observations
slope, intercept = np.polyfit(x, y, 1)
y_fit = slope * x + intercept
r2 = 1 - np.sum((y - y_fit) ** 2) / np.sum((y - y.mean()) ** 2)

# Plot a sample of points for performance
plot_sample = scatter_df.sample(n=min(100_000, len(scatter_df)), random_state=42)

fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(plot_sample["Pgen_pred"], plot_sample["Pgen"], alpha=0.2, s=8)

line_x = np.linspace(x.min(), x.max(), 100)
ax.plot(line_x, slope * line_x + intercept, color="red",
    label=f"Best fit: y = {slope:.3f}x + {intercept:.3f}\nR² = {r2:.3f}")

ax.set(
    title="Actual vs. Predicted PV Generation",
    xlabel="Predicted Pgen",
    ylabel="Actual Pgen",
)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()